#  Gold Layer - Payment Dimension

The Payment Dimension provides descriptive information about the payment methods used for taxi trips.

In the source dataset, payment methods are stored as numeric codes. While these codes are efficient for storage, they are not meaningful to business users.

This dimension converts those codes into readable descriptions, making dashboards and reports easier to understand.

### Source

Silver Layer (`taxi.silver.yellow_taxi`)

### Target

`taxi.gold.dim_payment`

In [0]:
from pyspark.sql import functions as F

silver_df = spark.table("taxi.silver.yellow_taxi")

In [0]:
dim_payment = (
    silver_df
    .select("payment_type")
    .distinct()
    .withColumnRenamed("payment_type", "payment_key")
)

In [0]:
dim_payment = (
    dim_payment
    .withColumn(
        "payment_description",
        F.when(F.col("payment_key") == 0, "Unknown")
         .when(F.col("payment_key") == 1, "Credit Card")
         .when(F.col("payment_key") == 2, "Cash")
         .when(F.col("payment_key") == 3, "No Charge")
         .when(F.col("payment_key") == 4, "Dispute")
         .when(F.col("payment_key") == 5, "Unknown")
         .when(F.col("payment_key") == 6, "Voided Trip")
         .otherwise("Other")
    )
)

In [0]:
display(
    dim_payment.orderBy("payment_key")
)

In [0]:
(
    dim_payment.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable("taxi.gold.dim_payment")
)

In [0]:
display(
    spark.table("taxi.gold.dim_payment")
)

In [0]:
%sql
DESCRIBE DETAIL taxi.gold.dim_payment;